# Tutorial: Análise de Cadeias de Valor com Knowledge Graph Federado

**V-ANTPC** - Value Chain Analysis with Networked Triple-Pattern Cognition

Este notebook demonstra como usar o sistema de Knowledge Graph para análise de cadeias de valor organizacionais.

---

## 📚 Índice

1. Setup e Conexão com Fuseki
2. Carregamento de Ontologias
3. Análise Econômica (e3value + REA)
4. Análise de Capacidades (VDML)
5. Análise de Supply Chain (SCOR)
6. Visualização de Grafos
7. Casos de Uso Avançados

---

## 1️⃣ Setup e Conexão com Fuseki

### Pré-requisitos

Antes de executar este notebook, certifique-se de que:

1. **Docker está instalado e rodando**
2. **Fuseki foi iniciado** via `./setup.sh`
3. **Porta 3030 está disponível**

Se ainda não executou o setup, abra um terminal e execute:

```bash
cd v-antpc
./setup.sh
```

In [103]:
# Importar bibliotecas necessárias
import sys
from pathlib import Path

# Adicionar diretório do projeto ao path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root / 'knowledge_graph'))

from value_chain_federator import ValueChainFederator

print("✓ Bibliotecas importadas com sucesso")

✓ Bibliotecas importadas com sucesso


In [104]:
# Inicializar o federator
fed = ValueChainFederator(
    endpoint="http://localhost:3030/kg",
    user="admin",
    password="admin"
)

print("Federador inicializado.")
print(f"Endpoint: {fed.endpoint}")

Federador inicializado.
Endpoint: http://localhost:3030/kg


In [105]:
# Verificar conectividade com Fuseki
if fed.verificar_conexao():
    print("\n✅ Conexão estabelecida com sucesso!")
    print("Você pode acessar a interface web em: http://localhost:3030")
else:
    print("\n❌ Não foi possível conectar ao Fuseki.")
    print("Verifique se o Docker está rodando e se executou ./setup.sh")

✓ Conectado ao Fuseki: http://localhost:3030/kg

✅ Conexão estabelecida com sucesso!
Você pode acessar a interface web em: http://localhost:3030


### ✅ Verificação de Namespaces

**IMPORTANTE:** O sistema precisa de namespaces customizados para trabalhar com dados.

Se você atualizou o código recentemente, **reinicie o kernel do Jupyter** para carregar a versão mais recente:

```
Kernel → Restart Kernel → Run All
```

In [ ]:
# Verificar namespaces disponíveis (incluindo namespaces customizados)
print("📋 Namespaces disponíveis no Federador:\n")

namespaces_core = ['e3', 'rea', 'vdml', 'scor']
namespaces_custom = ['actor', 'exchange', 'port', 'object', 'capability', 'metric']

print("Ontologias principais:")
for ns in namespaces_core:
    if ns in fed.namespaces:
        print(f"  ✓ {ns}: {fed.namespaces[ns]}")
    else:
        print(f"  ✗ {ns}: FALTANDO")

print("\nNamespaces customizados (para dados):")
for ns in namespaces_custom:
    if ns in fed.namespaces:
        print(f"  ✓ {ns}: {fed.namespaces[ns]}")
    else:
        print(f"  ✗ {ns}: FALTANDO - REINICIE O KERNEL!")

if all(ns in fed.namespaces for ns in namespaces_custom):
    print("\n✅ Todos os namespaces estão disponíveis!")
else:
    print("\n⚠️ ATENÇÃO: Namespaces customizados faltando!")
    print("   Execute: Kernel → Restart Kernel → Run All")

## 2️⃣ Carregamento de Ontologias

O sistema usa **4 ontologias complementares**:

- **e3value**: Trocas de valor e reciprocidade econômica
- **REA**: Recursos, Eventos, Agentes (dualidade econômica)
- **VDML**: Capacidades organizacionais e propostas de valor
- **SCOR**: Processos de supply chain e métricas de desempenho

In [106]:
# Carregar todas as ontologias
results = fed.carregar_todas_ontologias()

print("\n📊 Resumo do Carregamento:")
for ontologia, status in results.items():
    emoji = "✅" if status else "❌"
    print(f"{emoji} {ontologia}: {'Carregada' if status else 'Falhou'}")


=== Carregando Ontologias de Value Chain ===

✓ Ontologia 'e3value' carregada: 170 triplas no grafo http://valuechain.org/ontology/e3value
✓ Ontologia 'rea' carregada: 266 triplas no grafo http://valuechain.org/ontology/rea
✓ Ontologia 'vdml' carregada: 268 triplas no grafo http://valuechain.org/ontology/vdml
✓ Ontologia 'scor' carregada: 258 triplas no grafo http://valuechain.org/ontology/scor

=== Resumo do Carregamento ===
Sucesso: 4/4 ontologias

📊 Resumo do Carregamento:
✅ e3value: Carregada
✅ rea: Carregada
✅ vdml: Carregada
✅ scor: Carregada


In [107]:
# Estatísticas gerais do grafo
stats = fed.estatisticas_grafo()

print("\n📈 Estatísticas do Knowledge Graph:")
print(f"Total de triplas: {stats.get('total_triplas', 0):,}")
print(f"Total de classes: {stats.get('total_classes', 0)}")
print(f"Total de atores: {stats.get('total_atores', 0)}")
print(f"Total de capacidades: {stats.get('total_capacidades', 0)}")


📈 Estatísticas do Knowledge Graph:
Total de triplas: 1,417
Total de classes: 28
Total de atores: 8
Total de capacidades: 7


## 3️⃣ Análise Econômica (e3value + REA)

### 3.1 Listar Atores da Rede de Valor

In [108]:
# Listar todos os atores
atores = fed.listar_atores()

print(f"\n🏢 Total de atores na rede: {len(atores)}\n")

if atores:
    print("Atores identificados:")
    for ator in atores[:10]:  # Mostrar primeiros 10
        nome = ator['uri'].split('/')[-1]
        tipo = ator['tipo'].split('#')[-1]
        print(f"  • {nome} ({tipo})")
else:
    print("⚠️ Nenhum ator encontrado. Você precisa carregar dados de exemplo.")
    print("Execute a Seção 7 para inserir dados do APL Têxtil de PE.")


🏢 Total de atores na rede: 10

Atores identificados:
  • ConfecoesCaruaru (Manufacturer)
  • DistribuidoraNacional (Distributor)
  • EmbalagemNordeste (Supplier)
  • FeiraSulancaCaruaru (Distributor)
  • GrandesRedes (Customer)
  • LinhasElasticosSC (Supplier)
  • LinhasElasticosSC (Supplier)
  • RotaDoMar (Manufacturer)
  • TecelagemSP (Supplier)
  • TecelagemSP (Supplier)


### 3.2 Mapear Trocas de Valor

In [109]:
# Listar trocas de valor entre atores
trocas = fed.listar_trocas_valor()

print(f"\n🔄 Total de trocas de valor: {len(trocas)}\n")

if trocas:
    print("Exemplos de trocas:")
    for troca in trocas[:5]:
        provedor = troca['provedor'].split('/')[-1]
        receptor = troca['receptor'].split('/')[-1]
        objeto = troca['valor_objeto'].split('/')[-1]
        print(f"  {provedor} → {receptor}: {objeto}")
else:
    print("⚠️ Nenhuma troca encontrada. Carregue dados de exemplo primeiro.")


🔄 Total de trocas de valor: 6

Exemplos de trocas:
  TecelagemSP → RotaDoMar: tecidos_planos
  RotaDoMar → FeiraSulancaCaruaru: roupas_casual
  FeiraSulancaCaruaru → LojistasRegionais: roupas_atacado
  RotaDoMar → TecelagemSP: dinheiro_tecidos
  FeiraSulancaCaruaru → RotaDoMar: dinheiro_roupas


### 3.3 Calcular Lucratividade de um Ator

In [110]:
# Exemplo: calcular lucratividade de um ator específico
# URI correta: http://valuechain.org/data/actor/RotaDoMar

exemplo_ator_uri = "http://valuechain.org/data/actor/RotaDoMar"

resultado = fed.calcular_lucratividade_ator(exemplo_ator_uri)

if resultado:
    print(f"\n💰 Análise de Lucratividade - {exemplo_ator_uri.split('/')[-1]}\n")
    print(f"Receita Total: R$ {resultado['receita']:,.2f}")
    print(f"Custo Total:   R$ {resultado['custo']:,.2f}")
    print(f"Lucro Líquido: R$ {resultado['lucro']:,.2f}")
    print(f"Margem:        {resultado['margem_percentual']:.2f}%")
else:
    print(f"⚠️ Não foi possível calcular lucratividade para {exemplo_ator_uri}")
    print("Verifique se o ator existe e tem dados econômicos associados.")


💰 Análise de Lucratividade - RotaDoMar

Receita Total: R$ 0.00
Custo Total:   R$ 15,045.00
Lucro Líquido: R$ -15,045.00
Margem:        0.00%


### 3.4 Verificar Dualidade Econômica (REA)

O modelo REA exige que **todo evento de incremento tenha um decremento dual** (e vice-versa).

In [111]:
# Analisar pares de eventos duais
pares_duais = fed.analisar_dualidade_rea()

print(f"\n⚖️ Pares de Eventos Duais (REA): {len(pares_duais)}\n")

if pares_duais:
    for par in pares_duais[:3]:
        print(f"Incremento: {par['incremento'].split('/')[-1]}")
        print(f"  ↔ Decremento: {par['decremento'].split('/')[-1]}")
        print(f"Recurso: {par['recurso'].split('/')[-1]}")
        print(f"Quantidade: {par['quantidade']}")
        print("---")
else:
    print("⚠️ Nenhum par dual encontrado.")
    print("\nNOTA: A análise de dualidade REA requer dados específicos com:")
    print("  • rea:IncrementEvent (eventos de aumento de recurso)")
    print("  • rea:DecrementEvent (eventos de redução de recurso)")
    print("  • Propriedade rea:duality conectando os pares")
    print("\nOs dados atuais do APL Têxtil usam apenas e3value (trocas de valor).")
    print("Para testar dualidade REA, carregue dados com eventos de incremento/decremento.")


⚖️ Pares de Eventos Duais (REA): 0

⚠️ Nenhum par dual encontrado.

NOTA: A análise de dualidade REA requer dados específicos com:
  • rea:IncrementEvent (eventos de aumento de recurso)
  • rea:DecrementEvent (eventos de redução de recurso)
  • Propriedade rea:duality conectando os pares

Os dados atuais do APL Têxtil usam apenas e3value (trocas de valor).
Para testar dualidade REA, carregue dados com eventos de incremento/decremento.


## 4️⃣ Análise de Capacidades (VDML)

### 4.1 Listar Capacidades Organizacionais

In [112]:
# Listar todas as capacidades
capacidades = fed.listar_capacidades()

print(f"\n🎯 Total de capacidades: {len(capacidades)}\n")

if capacidades:
    for cap in capacidades[:10]:
        nome = cap['uri'].split('/')[-1]
        is_core = "CORE" if cap['is_core'] == 'true' else "SUPPORT"
        nivel = cap.get('nivel', 'N/A')
        print(f"  [{is_core}] {nome} (Nível: {nivel}/5)")
else:
    print("⚠️ Nenhuma capacidade encontrada.")


🎯 Total de capacidades: 7

  [CORE] design_moda (Nível: Alto/5)
  [CORE] logistica_distribuicao (Nível: Alto/5)
  [CORE] producao_rapida (Nível: Alto/5)
  [CORE] CoreCapability (Nível: N/A/5)
  [SUPPORT] gestao_estoque (Nível: Médio/5)
  [SUPPORT] relacionamento_fornecedores (Nível: Médio/5)
  [SUPPORT] SupportCapability (Nível: N/A/5)


### 4.2 Identificar Capacidades Core (Diferenciadoras)

In [113]:
# Filtrar apenas capacidades core
capacidades_core = fed.listar_capacidades(apenas_core=True)

print(f"\n🌟 Capacidades CORE (Diferenciadoras): {len(capacidades_core)}\n")

if capacidades_core:
    for cap in capacidades_core:
        print(f"✓ {cap['descricao']}")
        print(f"  Nível de Maturidade: {cap.get('nivel', 'N/A')}/5\n")
else:
    print("⚠️ Nenhuma capacidade core encontrada.")

✗ Erro na consulta SPARQL: QueryBadFormed: A bad request has been sent to the endpoint: probably the SPARQL query is badly formed. 

Response:
b'Parse error: Encountered " "filter" "FILTER "" at line 22, column 9.\n'

🌟 Capacidades CORE (Diferenciadoras): 0

⚠️ Nenhuma capacidade core encontrada.


## 5️⃣ Análise de Supply Chain (SCOR)

### 5.1 Listar Métricas SCOR

In [114]:
# Listar métricas de desempenho SCOR
metricas = fed.listar_metricas_scor()

print(f"\n📊 Métricas SCOR: {len(metricas)}\n")

if metricas:
    print(f"{'Métrica':<40} {'Atual':<15} {'Alvo':<15} {'Gap':<10}")
    print("-" * 80)
    
    for metrica in metricas[:10]:
        nome = metrica['metrica'].split('/')[-1]
        atual = metrica.get('valor_atual', 'N/A')
        alvo = metrica.get('valor_alvo', 'N/A')
        unidade = metrica.get('unidade', '')
        
        try:
            gap = float(alvo) - float(atual)
            print(f"{nome:<40} {atual} {unidade:<10} {alvo} {unidade:<10} {gap:.2f}")
        except (ValueError, TypeError):
            print(f"{nome:<40} {atual} {unidade:<10} {alvo} {unidade:<10} -")
else:
    print("⚠️ Nenhuma métrica SCOR encontrada.")


📊 Métricas SCOR: 10

Métrica                                  Atual           Alvo            Gap       
--------------------------------------------------------------------------------
cash_to_cash                             45.0 dias       30.0 dias       -15.00
cycle_time                               12.5 dias       8.0 dias       -4.50
perfect_order                            87.5 %          95.0 %          7.50
supply_chain_cost                        18.3 % da receita 15.0 % da receita -3.30
upward_flexibility                       25.0 % de aumento 35.0 % de aumento 10.00
scor#CashToCashCycleTime                 N/A N/A        N/A N/A        -
scor#OrderFulfillmentCycleTime           N/A N/A        N/A N/A        -
scor#PerfectOrderFulfillment             N/A N/A        N/A N/A        -
scor#SupplyChainCost                     N/A N/A        N/A N/A        -
scor#UpwardSupplyChainFlexibility        N/A N/A        N/A N/A        -


## 6️⃣ Consultas SPARQL Customizadas

### 6.1 Executar Query Personalizada

In [115]:
# Exemplo de query SPARQL customizada
query = """
SELECT ?ator ?nome (COUNT(?porta) AS ?numPortas)
WHERE {
    GRAPH ?g {
        ?ator a e3:Actor .
        OPTIONAL { ?ator rdfs:label ?nome }
        OPTIONAL { ?ator e3:has_value_port ?porta }
    }
}
GROUP BY ?ator ?nome
ORDER BY DESC(?numPortas)
LIMIT 10
"""

results = fed.consultar(query)

if results and 'results' in results:
    print("\n🔍 Atores por número de portas de valor:\n")
    
    bindings = results['results']['bindings']
    if bindings:
        for binding in bindings:
            ator = binding['ator']['value'].split('/')[-1]
            nome = binding.get('nome', {}).get('value', 'N/A')
            num_portas = binding['numPortas']['value']
            print(f"  {ator}: {num_portas} portas de valor")
    else:
        print("  Nenhum ator com portas de valor encontrado.")
        print("  Execute o script load_apl_data.py para carregar dados de exemplo.")
else:
    print("⚠️ Query não retornou resultados")


🔍 Atores por número de portas de valor:

  FeiraSulancaCaruaru: 4 portas de valor
  RotaDoMar: 4 portas de valor
  TecelagemSP: 2 portas de valor
  ConfecoesCaruaru: 0 portas de valor
  DistribuidoraNacional: 0 portas de valor
  EmbalagemNordeste: 0 portas de valor
  GrandesRedes: 0 portas de valor
  LinhasElasticosSC: 0 portas de valor


### 6.2 Carregar e Executar Query de Arquivo

In [121]:
# Exemplo de query SPARQL para análise econômica
# Query 1: Trocas de Valor com Valores Monetários

query = """
SELECT DISTINCT ?provedor ?nomeProvedor ?receptor ?nomeReceptor ?valorObjeto ?valor
WHERE {
    GRAPH ?g {
        # Troca de valor
        ?troca a e3:ValueExchange .
        ?troca e3:connects ?porta1, ?porta2 .
        
        # Ator provedor
        ?provedor e3:has_value_port ?porta1 .
        ?porta1 e3:direction "out" .
        ?porta1 e3:offers ?valorObjeto .
        OPTIONAL { ?provedor rdfs:label ?nomeProvedor }
        
        # Ator receptor
        ?receptor e3:has_value_port ?porta2 .
        ?porta2 e3:direction "in" .
        ?porta2 e3:requests ?valorObjeto .
        OPTIONAL { ?receptor rdfs:label ?nomeReceptor }
        
        # Valor econômico
        OPTIONAL { ?valorObjeto e3:economic_value ?valor }
    }
}
ORDER BY ?provedor ?receptor
LIMIT 10
"""

print("📄 Executando query de análise econômica...\n")

results = fed.consultar(query)

if results and 'results' in results:
    bindings = results['results']['bindings']
    
    if bindings:
        print(f"✓ Retornou {len(bindings)} trocas de valor:\n")
        
        for binding in bindings:
            prov = binding.get('nomeProvedor', {}).get('value', binding['provedor']['value'].split('/')[-1])
            rec = binding.get('nomeReceptor', {}).get('value', binding['receptor']['value'].split('/')[-1])
            obj = binding['valorObjeto']['value'].split('/')[-1]
            val = binding.get('valor', {}).get('value', 'N/A')
            
            print(f"  {prov} → {rec}: {obj} (R$ {val})")
    else:
        print("Nenhuma troca de valor encontrada.")
        print("Execute: python knowledge_graph/load_apl_data.py")
else:
    print("Query executada, mas sem resultados")

📄 Executando query de análise econômica...

✓ Retornou 6 trocas de valor:

  Feira da Sulanca Caruaru → Lojistas Regionais: roupas_atacado (R$ 50.0)
  Feira da Sulanca Caruaru → Rota do Mar - Confecções: dinheiro_roupas (R$ 45.0)
  Lojistas Regionais → Feira da Sulanca Caruaru: dinheiro_atacado (R$ 50.0)
  Rota do Mar - Confecções → Feira da Sulanca Caruaru: roupas_casual (R$ 45.0)
  Rota do Mar - Confecções → Tecelagem São Paulo SA: dinheiro_tecidos (R$ 15000.0)
  Tecelagem São Paulo SA → Rota do Mar - Confecções: tecidos_planos (R$ 15000.0)


## 7️⃣ Inserir Dados de Exemplo (APL Têxtil de PE)

### 7.1 Inserir Ator de Exemplo

In [122]:
# Exemplo de inserção de triplas RDF
# IMPORTANTE: Usando URIs completas para evitar problemas de prefixes
triplas_exemplo = """
<http://valuechain.org/data/actor/RotaDoMar_exemplo> a e3:Actor ;
    rdfs:label "Rota do Mar - Confecções (Exemplo)" ;
    rdfs:comment "Empresa de referência do APL Têxtil de Pernambuco, fundada em 1996" ;
    e3:profitability 50.3 .

<http://valuechain.org/data/actor/PoloToritama_exemplo> a e3:MarketSegment ;
    rdfs:label "Polo Toritama Jeans (Exemplo)" ;
    rdfs:comment "Agregação de 3000+ empresas especializadas em jeans" .
"""

sucesso = fed.inserir_triplas(
    triplas_exemplo,
    graph_uri="http://valuechain.org/data/apl-textil-pe"
)

if sucesso:
    print("✅ Dados de exemplo inseridos com sucesso!")
    print("\nNOTA: Para carregar o dataset completo do APL Têxtil PE, execute:")
    print("  python knowledge_graph/load_apl_data.py")
    print("\nDepois execute novamente as células de análise para ver os resultados.")
else:
    print("❌ Erro ao inserir dados")
    print("\nSe o erro persistir:")
    print("1. Reinicie o kernel do Jupyter: Kernel → Restart Kernel")
    print("2. Execute todas as células novamente desde o início")

✓ Triplas inseridas no grafo http://valuechain.org/data/apl-textil-pe
✅ Dados de exemplo inseridos com sucesso!

NOTA: Para carregar o dataset completo do APL Têxtil PE, execute:
  python knowledge_graph/load_apl_data.py

Depois execute novamente as células de análise para ver os resultados.


### 7.2 Verificar Inserção

In [123]:
# Re-listar atores para verificar inserção
atores_atualizados = fed.listar_atores()

print(f"\n🏢 Atores após inserção: {len(atores_atualizados)}")

if atores_atualizados:
    for ator in atores_atualizados:
        nome = ator['uri'].split('/')[-1]
        print(f"  • {nome}")


🏢 Atores após inserção: 10
  • ConfecoesCaruaru
  • DistribuidoraNacional
  • EmbalagemNordeste
  • FeiraSulancaCaruaru
  • GrandesRedes
  • LinhasElasticosSC
  • LinhasElasticosSC
  • RotaDoMar
  • TecelagemSP
  • TecelagemSP


## 8️⃣ Limpeza e Reset

### 8.1 Limpar Dados de Instância (Manter Ontologias)

In [119]:
# CUIDADO: Isso vai remover todos os dados de instância!
# Descomente a linha abaixo apenas se tiver certeza

# fed.limpar_grafo(graph_uri="http://valuechain.org/data/apl-textil-pe")

print("⚠️ Limpeza desabilitada por segurança.")
print("Descomente o código acima para executar.")

⚠️ Limpeza desabilitada por segurança.
Descomente o código acima para executar.


## 🎓 Conclusão

Você aprendeu a:

✅ Conectar ao Fuseki e carregar ontologias  
✅ Listar atores, trocas e capacidades  
✅ Calcular lucratividade de atores  
✅ Verificar dualidade econômica (REA)  
✅ Analisar capacidades core vs support  
✅ Executar queries SPARQL customizadas  
✅ Inserir dados de exemplo  

---

### 📚 Próximos Passos

1. **Carregar datasets completos**: Use os arquivos em `data/apl_textil_pe_parte*.md`
2. **Explorar queries avançadas**: Veja `knowledge_graph/sparql_queries/`
3. **Visualizar grafos**: Implemente visualizações com PyVis
4. **Criar análises customizadas**: Combine múltiplas ontologias

---

**Documentação**: Veja `README.md` e `QUICKSTART.md` para mais detalhes.
